# MRS Quality Control

Checks data-quality metrics for the MEGA-PRESS MRS acquisitions (GABA+/Glx in V1/EVC and
M1/SMC) before they're trusted for the main analysis: NAA linewidth (FWHM), signal-to-noise
ratio (SNR), and Cramer-Rao Lower Bounds (%CRLB) on the metabolite fits, each compared
pre- vs post-training. Also checks MRS voxel placement consistency between sessions
(percent overlap) and lists further QC metrics that could be added later.

In [ ]:
import importlib
import import_packages # all my packages are there, it's easier
importlib.reload(import_packages) # so I can add new packages in import_packages.py without restarting the kernel
from import_packages import *
print("Packages (re)loaded successfully!")

In [ ]:
# Set Up the paths 
ori = os.getcwd()
path_data_ref = os.path.abspath(os.path.join(ori, "..", "Data", "Raw"))
path_plot = os.path.abspath(os.path.join(ori, "..", "Data", "Plots"))

In [ ]:
mrs_qc = pd.read_csv(path_data_ref+'/combined_qc_diff.csv') 
# mrs_tf = pd.read_csv(path_data_ref+'/combined_tissue_fraction.csv') 

mrs_qc_edit_off = pd.read_csv(path_data_ref+'/combined_qc_edit_off.csv') 


# first the linewidth of NAA ofr edit-off

It should be between 5-10Hz, the larger, the worse (all good for us, it is mostly at 5 or below)

In [ ]:
mrs_qc_ref = mrs_qc_edit_off.copy()
mrs_qc_ref = mrs_qc_ref[(mrs_qc_ref['Metab'] == 'NAA')]

In [ ]:
print(mrs_qc_ref.head())


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
# Ensure correct ordering
mrs_qc_ref["session"] = pd.Categorical(
    mrs_qc_ref["session"],
    categories=["pre", "post"],
    ordered=True
)

mrs_qc_ref["location"] = pd.Categorical(
    mrs_qc_ref["location"],
    categories=["M1", "V1"],
    ordered=True
)


In [ ]:
location_name_map = {'V1': 'EVC', 'M1': 'SMC'}

ax = mrs.plot_paired_sessions(
    mrs_qc_ref, requested_label='EVC', y_col='FWHM', y_label='NAA FWHM',
    id_col='participant_number', location_name_map=location_name_map, ylim=(3.5, 7)
)
ax.text(0.74, 5.25, "n.s.", ha='center', va='bottom', color='black', fontsize=18)
plt.savefig('../Data/Plots/new_new/EVC_NAA_FWHM.png', dpi=600, bbox_inches='tight')
plt.show()

ax = mrs.plot_paired_sessions(
    mrs_qc_ref, requested_label='SMC', y_col='FWHM', y_label='NAA FWHM',
    id_col='participant_number', location_name_map=location_name_map, ylim=(3.5, 7)
)
ax.text(0.74, 5.25, "n.s.", ha='center', va='bottom', color='black', fontsize=18)
plt.savefig('../Data/Plots/new_new/SMC_NAA_FWHM.png', dpi=600, bbox_inches='tight')
plt.show()


remember to do all the measures again but with edit-off values 

# Signal-to-Noise Ratio (SNR) for NAA

In [ ]:
mrs_qc = pd.read_csv(path_data_ref+'/combined_qc_diff.csv') 

mrs_qc_ref = mrs_qc_edit_off.copy()
mrs_qc_ref = mrs_qc_ref[(mrs_qc_ref['Metab'] == 'GABA')]

# Ensure correct ordering
mrs_qc_ref["session"] = pd.Categorical(
    mrs_qc_ref["session"],
    categories=["pre", "post"],
    ordered=True
)

mrs_qc_ref["location"] = pd.Categorical(
    mrs_qc_ref["location"],
    categories=["M1", "V1"],
    ordered=True
)
print(mrs_qc_ref.head())


In [ ]:
ax = mrs.plot_paired_sessions(
    mrs_qc_ref, requested_label='EVC', y_col='SNR', y_label='NAA SNR',
    id_col='participant_number', location_name_map=location_name_map
)
plt.savefig('../Data/Plots/EVC_NAA_SNR.png', dpi=600, bbox_inches='tight')
plt.show()

ax = mrs.plot_paired_sessions(
    mrs_qc_ref, requested_label='SMC', y_col='SNR', y_label='NAA SNR',
    id_col='participant_number', location_name_map=location_name_map
)
plt.savefig('../Data/Plots/SMC_NAA_SNR.png', dpi=600, bbox_inches='tight')
plt.show()


### let's go back to main measures

# Cramér–Rao Lower Bounds (CRLB) or metabolites (individual and combined)

I should be below 20% with an acceptable range of 20-30 % for very small quantites

In [ ]:
mrs_qc = pd.read_csv(path_data_ref+'/combined_summary_diff.csv') 



# Ensure correct ordering
mrs_qc["session"] = pd.Categorical(
    mrs_qc["session"],
    categories=["pre", "post"],
    ordered=True
)

mrs_qc["location"] = pd.Categorical(
    mrs_qc["location"],
    categories=["M1", "V1"],
    ordered=True
)




In [ ]:
mrs_qc = mrs_qc[(mrs_qc['Metab'] == 'NAA')]


In [ ]:
ax = mrs.plot_paired_sessions(
    mrs_qc, requested_label='EVC', y_col='%CRLB', y_label='NAA %CRLB',
    id_col='participant_number', location_name_map=location_name_map, ylim=(1, 3)
)
plt.savefig('../Data/Plots/EVC_NAA_CRLB.png', dpi=600, bbox_inches='tight')
plt.show()

ax = mrs.plot_paired_sessions(
    mrs_qc, requested_label='SMC', y_col='%CRLB', y_label='NAA %CRLB',
    id_col='participant_number', location_name_map=location_name_map, ylim=(1, 3)
)
plt.savefig('../Data/Plots/SMC_NAA_CRLB.png', dpi=600, bbox_inches='tight')
plt.show()


# Frequency and Phase Stability

Important especially for averaged spectra and edited MRS.

## Frequency Drift
What it measures: Temporal shift of resonance frequency across averages
Why it matters: Causes line broadening and subtraction artifacts
Typical acceptable range: < 2–5 Hz across acquisition

# Water-Related QC Metrics
## Water Suppression Efficiency

What it measures: Residual water signal relative to unsuppressed water scan
Why it matters: Poor suppression increases baseline distortions

## Unsuppressed Water SNR / Linewidth

Often used as a proxy QC measure, especially in large studies
Reflects both hardware and shimming quality

# Optional / Advanced QC Metrics


- Macromolecule contamination
- Editing efficiency (MEGA-PRESS)
- GABA or Glx-specific CRLB
- B₀ and B₁ field maps
- Voxel tissue composition (GM/WM/CSF fractions)

In [ ]:
path_data_refined = os.path.abspath(os.path.join(ori, "..", "Data", "Refined"))

mrs_voxels = pd.read_csv(path_data_refined+'/voxel_overlap.csv', header=0,index_col=False) 

print(mrs_voxels.columns.tolist())

In [ ]:
mrs_voxels["percent_overlap"] = (mrs_voxels[" Ses2_Overlap_Size"]/((mrs_voxels[" Ses1_Voxel_Size"]+mrs_voxels[" Ses2_Voxel_Size"])/2)*100)

In [ ]:
import statistics as st
print(st.median(mrs_voxels["percent_overlap"]))
print(st.mean(mrs_voxels["percent_overlap"]))
print(st.stdev(mrs_voxels["percent_overlap"]))
